# 05 — Advanced hybrid stacking, evaluation & export
**Level: advanced.** Execute the real NoHackFlow trainer only after the dataset audit and group split pass. Outputs are *not* production validated until Windows capture parity and external data tests pass.

In [ ]:
import os
from pathlib import Path
import json
from nohack.training.hybrid import train_from_splits
splits = Path(os.environ.get("NOHACK_SPLITS", "data/processed/nohack_splits"))
models = Path(os.environ.get("NOHACK_MODEL_DIR", "artifacts/models/nohack-v1"))
if not (splits / "manifest.json").exists():
    raise FileNotFoundError("First run notebook 03 on verified canonical data")
report = train_from_splits(splits, models, folds=4, seed=42, max_false_positive_rate=0.02)
print("Reported status:", report["model_status"])
print("Model digest:", report["bundle_sha256"])
print(json.dumps(report["metrics"], indent=2))

In [ ]:
from nohack.inference.hybrid import load_bundle, predict_features
from nohack.features import FEATURE_NAMES
import pandas as pd
bundle = load_bundle(models, trust_local_artifact=True)  # Only YOUR OWN trained artifact
heldout = pd.read_parquet(splits / "test.parquet")
example = {col: heldout.iloc[0][col].item() if hasattr(heldout.iloc[0][col], "item") else heldout.iloc[0][col] for col in FEATURE_NAMES}
result = predict_features(bundle, example)
print(json.dumps(result, indent=2))
assert result["assessment_scope"] == "network_flow_not_browser_tab"

In [ ]:
print("Export THESE FILES as one trusted bundle:")
for name in ("hybrid.joblib", "manifest.json", "metrics.json"):
    path = models / name
    print(name, path.exists(), path.stat().st_size if path.exists() else "")
print("Do not upload private training data, raw PCAP or local history as artifacts.")

### Final review gate
Record group-disjoint held-out metrics, false positive rate, calibration, attack-family macro-F1, drift, inference speed, Windows live parity and explicit limitations. Track experiment provenance with the MLflow import utility. A synthetic-test model cannot be advertised as a validated detector.